## Running kinGEMs Pipeline on $E. coli$ iML1515 Model

In [1]:

%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import cobra # type: ignore
import numpy as np # type: ignore
from datetime import datetime
import random
from cobra.io import write_sbml_model # type: ignore
import cobra as cb

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

# Import kinGEMs components
import kinGEMs
from kinGEMs.dataset import load_model, convert_to_irreversible, prepare_model_data, retrieve_sequences, map_metabolites, merge_substrate_sequences, process_merged_data_with_folds, process_kcat_predictions, assign_kcats_to_model, format_kcats_like_gpr, annotate_model_with_kcat_and_gpr
from kinGEMs.modeling.optimize import run_optimization_with_dataframe, validate_enzyme_constraints
from kinGEMs.modeling.tuning import simulated_annealing, analyze_kcat_changes
from kinGEMs.modeling.fva import flux_variability_analysis, plot_flux_variability, plot_cumulative_fvi_distribution


2025-06-09 14:33:12.123 | INFO     | kinGEMs.config:<module>:12 - PROJ_ROOT path is: C:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2


### Set your GEM name here

In [17]:

# === Configuration ===
organism_strain_GEMname = "ecoli_iML1515" # Update this
organism = "E coli" # Update this 
run_id = f"{organism_strain_GEMname}_{datetime.today().strftime('%Y%m%d')}_{random.randint(1000, 9999)}"

# Directories
data_dir = "../data"
raw_data_dir = os.path.join(data_dir, "raw")
interim_data_dir = os.path.join(data_dir, "interim")
interim_data_dir_spec = os.path.join(interim_data_dir, f"{organism_strain_GEMname}")
processed_data_dir = os.path.join(data_dir, "processed")
processed_data_dir_spec = os.path.join(processed_data_dir, f"{organism_strain_GEMname}")
CPIPred_data_dir = os.path.join(interim_data_dir, "CPI-Pred predictions")
results_dir = os.path.join(os.getcwd(), "results")
tuning_results_dir = os.path.join(results_dir, "tuning_results", run_id)
os.makedirs(tuning_results_dir, exist_ok=True)

# Input files
model_path = os.path.join(raw_data_dir, "iML1515_GEM.xml") # Update this
predictions_csv_path = os.path.join(CPIPred_data_dir, f"X06A_kinGEMs_{organism_strain_GEMname}_predictions copy.csv")

# Output files
substrates_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_substrates.csv")
sequences_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_sequences.csv")
merged_data_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_merged_data.csv")
processed_data_output = os.path.join(processed_data_dir_spec, f"{organism_strain_GEMname}_processed_data.csv")

# Simulation parameters
biomass_reaction = 'BIOMASS_Ec_iML1515_WT_75p37M' # Update this
enzyme_upper_bound = 0.5


### Step 1: Preparing and processing model data

In [3]:

print("=== Step 1: Preparing model data ===")
irrev_model, substrate_df, sequences_df = prepare_model_data(
    model_path=model_path,
    substrates_output=substrates_output,
    sequences_output=sequences_output,
    organism=organism
)


=== Step 1: Preparing model data ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-02-25
Loaded model with 2712 reactions and 1877 metabolites
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp20r7756v.lp
Reading time = 0.02 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Number of reactions that are non-exchange:  2381
Number of reactions that are exchange:  331
Number of reactions being added from non-exchange: 639
Number of reactions being added from exchange: 970
Converted to irreversible model with 3682 reactions
Extracted 6194 substrate-reaction pairs


c:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2\kinGEMs\dataset.py:215: DtypeWarning: Columns (4,10) have mixed types. Specify dtype option on import or set low_memory=False.
  SEED_comps = pd.read_csv(SEED_COMPOUNDS, sep='\t')
2025-06-03 13:57:47,483 - kinGEMs.dataset - INFO - There are 1814 substrates in the GEM.
2025-06-03 13:57:47,515 - kinGEMs.dataset - INFO - -----------------------------
2025-06-03 13:57:47,515 - kinGEMs.dataset - INFO - Mapping substrate: ala__D_c
2025-06-03 13:57:47,546 - kinGEMs.dataset - INFO - BiGG Name: D-Alanine
2025-06-03 13:57:47,589 - kinGEMs.dataset - INFO - SMILES found in SEED: C[C@@H]([NH3+])C(=O)[O-]
2025-06-03 13:57:47,590 - kinGEMs.dataset - INFO - -----------------------------
2025-06-03 13:57:47,590 - kinGEMs.dataset - INFO - Mapping substrate: pydx5p_c
2025-06-03 13:57:47,617 - kinGEMs.dataset - INFO - BiGG Name: Pyridoxal 5'-phosphate
2025-06-03 13:57:47,657 - kinGEMs.dataset - INFO - SMILES found in MetaNetX: Cc1ncc(COP(=O)([O-])[O-])c(C

Mapped metabolites to SMILES (5686 found)


2025-06-03 14:05:46,792 - root - WARNING - No sequence found for gene s0001


Retrieved 1515 protein sequences


### Step 2: Merging substrate and sequence data

In [18]:

print("=== Step 2: Merging substrate and sequence data ===")

substrate_df = pd.read_csv(substrates_output)
sequences_df = pd.read_csv(sequences_output)
model = load_model(model_path)
irrev_model = convert_to_irreversible(model)

merged_data = merge_substrate_sequences(
    substrate_df=substrate_df,
    sequences_df=sequences_df,
    model=irrev_model,
    output_path=merged_data_output
)


=== Step 2: Merging substrate and sequence data ===
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp9sw0863l.lp
Reading time = 0.02 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
Number of reactions that are non-exchange:  2381
Number of reactions that are exchange:  331
Number of reactions being added from non-exchange: 639
Number of reactions being added from exchange: 970


### Step 3: Processing CPI-Pred kcat values

In [23]:

print("=== Step 3: Processing CPI-Pred kcat values & annotating model ===")
processed_data = process_kcat_predictions(
    merged_df=merged_data,
    predictions_csv_path=predictions_csv_path,
    output_path=processed_data_output
)

irrev_model = annotate_model_with_kcat_and_gpr(
    model=irrev_model,
    df=processed_data
)

print("=== Step 4: Running optimization with kcat constraints ===")
(solution_value, df_FBA, gene_sequences_dict, _)=run_optimization_with_dataframe(
    model=irrev_model, 
    processed_df=processed_data, 
    objective_reaction=biomass_reaction, 
    enzyme_upper_bound=enzyme_upper_bound, 
    enzyme_ratio=True, 
    maximization=True, 
    multi_enzyme_off=False, 
    isoenzymes_off=False, 
    promiscuous_off=False, 
    complexes_off=False,
    output_dir=None, 
    save_results=False,
    print_reaction_conditions=True)

print("Biomass value: ", solution_value)





=== Step 3: Processing CPI-Pred kcat values & annotating model ===
=== Step 4: Running optimization with kcat constraints ===
Loaded model from input irreversible model!
PRINT INITIAL KCAT DICT:  {}
OBJECTIVE FUNCTION IS:  reaction[BIOMASS_Ec_iML1515_WT_75p37M]
ERROR: evaluating object as numeric value: enzyme[b4139]
        (object: <class 'pyomo.core.base.var._GeneralVarData'>)
    No value for uninitialized NumericValue object enzyme[b4139]
An error occurred during optimization: No value for uninitialized NumericValue object enzyme[b4139]
Solver used: <undefined>
Termination condition: optimal
Solver status: ok
Biomass flux: 0.00010313029087294376
Constraint block: set_S_mat
  set_S_mat[btn_e]:
     expr : - reaction[BTNtex] + reaction[BTNtex_reverse] - reaction[EX_btn_e] + reaction[EX_btn_e_reverse]
     lower: 0.0   upper: 0.0
  set_S_mat[23dhbzs3_c]:
     expr : 2.0*reaction[FE3DHBZS3R]
     lower: 0.0   upper: 0.0
  set_S_mat[mmet_e]:
     expr : reaction[EX_mmet_e_reverse] + re

In [12]:
from Bio.SeqUtils.ProtParam import molecular_weight

# 1) Reconstruct kcat_dict_original exactly as run_optimization_with_dataframe did:
#    It looked for rows where kcat_mean and SEQ were not NaN, then did:
#      kcat_dict[(reaction_id, gene_id)] = [kcat_mean]
#
kcat_dict_original = {}
for _, row in processed_data.iterrows():
    if pd.notna(row['kcat_mean']) and pd.notna(row['SEQ']):
        rxn_id  = row['Reactions']
        gene_id = row['Single_gene']
        kcat_s  = float(row['kcat_mean'])  # this is in 1/s
        # store as a list (their function always wrapped kcat in a list)
        kcat_dict_original[(rxn_id, gene_id)] = [kcat_s]

# 2) Convert each kcat from 1/s → 1/hr (multiply by 3600), matching run_optimization’s logic:
kcat_dict_hr = {}
for (rxn, gene), lst in kcat_dict_original.items():
    kcat_dict_hr[(rxn, gene)] = [val * 3600 for val in lst]


# 3) Rebuild reaction_gene_list (list of all (reaction, gene) pairs in the COBRA model):
reaction_gene_list = [
    (rxn.id, gene.id)
    for rxn in irrev_model.reactions
    for gene in rxn.genes
]


# 4) Rebuild gpr_dict exactly as run_optimization did:
#    It looked up reaction.annotation['gpr_replaced'] and stored that if it existed.
gpr_dict = {}
for rxn in irrev_model.reactions:
    gr = rxn.annotation.get('gpr_replaced')
    if gr is not None:
        gpr_dict[rxn.id] = gr


# 5) Rebuild S_mat = {(met_id, rxn_id): stoichiometric_coefficient}:
S_mat = {}
for met in irrev_model.metabolites:
    for rxn in met.reactions:
        coeff = rxn.get_coefficient(met.id)
        if coeff != 0:
            S_mat[(met.id, rxn.id)] = coeff


# 6) Rebuild enzyme_mw_dict exactly as run_optimization did:
#    Inside run_optimization, they used:
#      def calculate_molecular_weight(sequence):
#          if not sequence: return 0
#          try:  return molecular_weight(sequence, seq_type='protein')
#          except: return 0
#      ...
#      if gene_id in gene_sequences_dict and gene_sequences_dict[gene_id]:
#          mw = calculate_molecular_weight(seq)
#          enzyme_mw_dict[gene_id] = mw if mw>0 else 100000
#      else:
#          enzyme_mw_dict[gene_id] = 100000

def calculate_molecular_weight(seq: str) -> float:
    if not seq:
        return 0.0
    try:
        return molecular_weight(seq, seq_type='protein')
    except:
        return 0.0

enzyme_mw_dict = {}
for gene_id, seq in gene_sequences_dict.items():
    if seq:
        mw = calculate_molecular_weight(seq)
        enzyme_mw_dict[gene_id] = mw if mw > 0.0 else 100000.0
    else:
        enzyme_mw_dict[gene_id] = 100000.0


# 7) enzyme_ratio / enzyme_upper_bound are exactly what you passed in:
enzyme_ratio       = True               # you set enzyme_ratio=True
enzyme_upper_bound = enzyme_upper_bound # same float (e.g. 0.125)


# ─────────────────────────────────────────────────────────────────────────────
# 8) Now call validate_enzyme_constraints(...) with all reconstructed inputs:
#    (Assuming you have the validate_enzyme_constraints function defined as before.)
# ─────────────────────────────────────────────────────────────────────────────

validate_enzyme_constraints(
    df_FBA=df_FBA,
    kcat_dict_hr=kcat_dict_hr,
    gene_sequences_dict=gene_sequences_dict,
    reaction_gene_list=reaction_gene_list,
    gpr_dict=gpr_dict,
    enzyme_ratio=enzyme_ratio,
    enzyme_upper_bound=enzyme_upper_bound,
    enzyme_mw_dict=enzyme_mw_dict,
    S_mat=S_mat
)

All (reaction, gene) v_j ≤ kcat·e_i constraints are satisfied (within 1e-6 tolerance).
Total‐enzyme weight = 0.499339 ≤ 0.500000 (OK)
All metabolites satisfy steady‐state (S·v≈0).


### Step 4: Simulated Annealing

In [71]:

print("=== Step 4: Running simulated annealing ===")
temperature = 1.0
cooling_rate = 0.95
min_temperature = 0.01
max_iterations = 100
max_unchanged_iterations = 10
change_threshold = 0.001
biomass_goal = 0.2

kcat_dict, top_targets, df_new, iterations, biomasses, df_FBA = simulated_annealing(
    model=irrev_model,
    processed_data=processed_data,
    biomass_reaction=biomass_reaction,
    objective_value=biomass_goal,
    gene_sequences_dict=gene_sequences_dict,      # ← new
    output_dir=tuning_results_dir,
    enzyme_fraction=enzyme_upper_bound,
    temperature=temperature,
    cooling_rate=cooling_rate,
    min_temperature=min_temperature,
    max_iterations=max_iterations,
    max_unchanged_iterations=max_unchanged_iterations,
    change_threshold=change_threshold
)

print(f"Final biomass: {biomasses[-1]:.4f}")
print(f"Improvement: {(biomasses[-1] - biomasses[0]) / biomasses[0] * 100:.1f}%")
print("Top‐25 enzymes by mass contribution:")
print(top_targets[['Reactions','Single_gene','enzyme_mass']])


=== Step 4: Running simulated annealing ===
Loaded model from input irreversible model!
An error occurred during optimization: Solver did not find an optimal solution. Status: ok, Termination: optimal
Solver used: <undefined>
Termination condition: optimal
Solver status: ok
Biomass flux: 0.00010315816149778151


KeyError: 'Variable'

### Step 5: FVA 

In [ ]:

print("=== Step 5: Running Flux Variability Analysis ===")
fva_results_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_results.csv")
fva_plot_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_flux_range_plot.png")
fva_cumulative_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_cumulative_plot.png")

fva_results, _, _ = flux_variability_analysis(
    model=irrev_model,
    processed_df=df_new,
    biomass_reaction=biomass_reaction,
    output_file=fva_results_path,
    enzyme_upper_bound=enzyme_upper_bound
)

# Plot standard FVA range
fig = plot_flux_variability(fva_results, output_file=fva_plot_path)

# Plot cumulative distribution
plot_cumulative_fvi_distribution(
    dfs=[fva_results],
    labels=[organism_strain_GEMname],
    output_file=fva_cumulative_path
)


### Step 6: Save Final Model

In [ ]:
# Define output path for final GEM
model_output_dir = os.path.join("models")
os.makedirs(model_output_dir, exist_ok=True)
model_output_path = os.path.join(model_output_dir, f"{run_id}.xml")

# After simulated annealing
model_with_kcats = assign_kcats_to_model(irrev_model, df_new)

# Preview:
format_kcats_like_gpr(model_with_kcats.reactions.get_by_id("PGI"))


# Save the final irreversible model
write_sbml_model(model_with_kcats, model_output_path)

print(f"Final GEM saved to: {model_output_path}")